# Prompt chaining

This pattern breaks down large, complex tasks into smaller and more manageable problems. It's also known as Pipeline pattern and it's similar in nature to traditional orchestration: the output generated by one prompt is passed as input to the next promt in the chain.

In [ ]:
#First, prepare your environment
!uv pip install flyte. langchain-openai

58.79s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Using Python 3.12.12 environment at: /Users/davidmirror/code/agentic-patterns-on-v2/.venv
Audited 1 package in 10ms


Flyte allows you to tune compute resources (GPU, CPU, memory, ephemeral storage) via `Resources`. Caching is enabled to save cost and drive down execution times for subsequent runs by avoiding LLM calls with the same inputs.   

In [ ]:
import asyncio
import flyte
from flyte import TaskEnvironment, Resources
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

env = TaskEnvironment(
    name="llm_specs_env",
    resources=Resources(cpu="1", memory="1Gi"),
    cache="auto",  # cache LLM calls by inputs
)

This example uses `LangChain` with `ChatOpenAI` to:  

- Prompt 1: extract technical specs from text_input.
- Prompt 2: transform those specs into a JSON-like structure.

In [ ]:


llm = ChatOpenAI(temperature=0)  # constructed once per container

prompt_extract = ChatPromptTemplate.from_template(
    "Extract the technical specifications from the following text:\n\n{text_input}"
)
prompt_transform = ChatPromptTemplate.from_template(
    "Transform the following specifications into a JSON object with 'cpu', 'memory', and 'storage' as keys:\n\n{specifications}"
)

extraction_chain = prompt_extract | llm | StrOutputParser()
transform_chain = prompt_transform | llm | StrOutputParser()

The two-promt chain is implemented as two tasks where each task will run remotely on a container (or locally in a Python process) with its own compute resources: 



In [ ]:
@env.task
async def extract_specs(text_input: str) -> str:
    return await extraction_chain.ainvoke({"text_input": text_input})

@env.task
async def specs_to_json(specifications: str) -> str:
    return await transform_chain.ainvoke({"specifications": specifications})


In Flyte V2 you can have tasks calling other tasks, giving you the flexibility to write orchestration logic in pure Python.

In [ ]:
@env.task
async def process_description(text_input: str) -> str:
    specs = await extract_specs(text_input)
    json_str = await specs_to_json(specs)
    return json_str

**Rule of thumb:** Use this pattern when a task is too complex for a single prompt, involves multiple distinct processing stages, requires interaction with external tools between steps, or when building Agentic systems that need to perform multi-step reasoning and maintain state.

## Scaling the pattern

- Wrap the LLM calls with `@flyte.trace` if you want fine-grained traces and checkpointing around the LLM steps, so failed runs can resume without redoing all work. [Learn more](https://www.union.ai/docs/v2/byoc/user-guide/task-programming/traces/)
- Use task-level `retries` for transient LLM/API errors, and `timeout` to guard against hung requests. [Learn more](https://www.union.ai/docs/v2/byoc/user-guide/task-configuration/retries-and-timeouts/)
- Make it batchable: add a task that handles many product descriptions in parallel, using async patterns:


In [ ]:
@env.task
async def process_many_descriptions(descriptions: list[str]) -> list[str]:
    tasks = [process_description(text) for text in descriptions]
    results = await asyncio.gather(*tasks)
    return list(results)